# Mcardiac_EA epicardial ablation

Remove selected epicardial cells from the starting tissue and run simulation.


## Settings


In [1]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys

CWD = Path.cwd().resolve()
candidates = [CWD, CWD / "perturb", CWD / "experiments/Mcardiac_EA/perturb"]
PERTURB_DIR = next(p for p in candidates if (p / "config.json").is_file())
SETTINGS = json.loads((PERTURB_DIR / "config.json").read_text())
EXPERIMENT_DIR = PERTURB_DIR.parent
sys.path.insert(0, str(EXPERIMENT_DIR))
from workflow import read_training_manifest
MANIFEST = (PERTURB_DIR / SETTINGS["training_manifest"]).resolve()

DEVICE = "cuda:0"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

In [2]:
manifest = read_training_manifest(MANIFEST)
print("Route:", manifest["source"], "to", manifest["target"])
print("Simulation steps:", manifest["steps"])

Route: E10.5_CD1 to E12.5_CD1
Simulation steps: 20


## Set ablation


In [3]:
FRACTION = SETTINGS["fractions"][0]
SEED = SETTINGS["seeds"][0]
RUN_NAME = f"fraction{FRACTION:g}_seed{SEED}_{RUN_ID}"
OUTPUT_ROOT = (PERTURB_DIR / SETTINGS["output_root"]).resolve()
command = [
    sys.executable, str(PERTURB_DIR / "simulation.py"),
    "--training-manifest", str(MANIFEST), "--output-root", str(OUTPUT_ROOT),
    "--out-subdir", RUN_NAME, "--ablation-fraction", str(FRACTION),
    "--seed", str(SEED), "--latent-update-gain", str(SETTINGS["latent_update_gain"]),
    "--birth-hazard-max", "0.5", "--death-hazard-max", "0.04",
    "--message-passing-substeps", "3", "--coord-step-fraction", "0.15",
    "--ablation-only", "--device", DEVICE,
]
print(" ".join(command))


/home/zhouyj/.conda/envs/3dslice/bin/python /home/zhouyj/stVirtual_tutorial/experiments/Mcardiac_EA/perturb/simulation.py --training-manifest /home/zhouyj/stVirtual_tutorial/experiments/Mcardiac_EA/artifacts/training/E10.5_CD1_to_E12.5_CD1.json --output-root /home/zhouyj/stVirtual_tutorial/experiments/Mcardiac_EA/artifacts/extrapolation --out-subdir fraction0.2_seed2026_20261006_142935_126603 --ablation-fraction 0.2 --seed 2026 --latent-update-gain 1.2 --birth-hazard-max 0.5 --death-hazard-max 0.04 --message-passing-substeps 3 --coord-step-fraction 0.15 --ablation-only --device cuda:0


## Run simulation


In [4]:
result = subprocess.run(command, cwd=PERTURB_DIR, check=True, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print("Output directory:", OUTPUT_ROOT / RUN_NAME)

[Grid] H=64, W=72, H×W=4608
[auto cap] 3 {'src': {'q': 0.9, 'quantile_value': 1.0, 'mean': 1.0869060754776, 'max': 3.0, 'n_cells_used': 863}, 'tgt': {'q': 0.9, 'quantile_value': 3.0, 'mean': 2.219219207763672, 'max': 6.0, 'n_cells_used': 2331}, 'cap0': 1, 'capT': 3, 'cap': 3}
[diff] loaded /home/zhouyj/stVirtual_tutorial/experiments/Mcardiac_EA/transition_prior.csv Kmax= 1
{
  "training_manifest": "/home/zhouyj/stVirtual_tutorial/experiments/Mcardiac_EA/artifacts/training/E10.5_CD1_to_E12.5_CD1.json",
  "ablation_mode": "initial",
  "out_subdir": "fraction0.2_seed2026_20261006_142935_126603",
  "output_root": "/home/zhouyj/stVirtual_tutorial/experiments/Mcardiac_EA/artifacts/extrapolation",
  "ablation_fraction": 0.2,
  "birth_hazard_max": 0.5,
  "death_hazard_max": 0.04,
  "growth_target_count": 0,
  "latent_noise": 0.005,
  "lr_latent_alpha": 0.0,
  "latent_step_fraction": 0.08,
  "coord_step_fraction": 0.15,
  "message_passing_substeps": 3,
  "latent_update_gain": 1.2,
  "ablation_o